<a href="https://colab.research.google.com/github/RobotMa/UniAD/blob/v2.0-qianli/UniAD_Eval_Colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# UniAD 2.0 Evaluation on Google Colab

This notebook runs UniAD Stage 1 (Track & Map) evaluation on Colab's free T4 GPU.

**Setup:**
- Uses Colab's pre-installed PyTorch (no reinstall needed)
- Builds mmcv, mmdet, mmseg, mmdet3d from source for compatibility
- Caches packages to Google Drive for faster subsequent runs

**Requirements:**
- Google account with Google Drive
- nuScenes dataset (or mini version for testing)

**Estimated Time:**
- First run: ~15-20 min setup + 2-4 hours eval
- Subsequent runs: ~2-5 min setup (cached) + 2-4 hours eval

## 1. Check GPU

In [ ]:
# Verify GPU is available
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 2. Mount Google Drive

In [ ]:
import os
from google.colab import drive

# Mount Google Drive (handles already-mounted case gracefully)
if os.path.ismount('/content/drive'):
    print("Google Drive already mounted at /content/drive")
else:
    drive.mount('/content/drive')
    print("Google Drive mounted successfully!")

# Create cache directories on Google Drive for persistence
!mkdir -p /content/drive/MyDrive/colab_cache/pip
!mkdir -p /content/drive/MyDrive/colab_cache/UniAD_ckpts
print("Cache directories ready.")

## 3. Clone UniAD Repository

In [ ]:
import os
import subprocess

%cd /content

# Clone UniAD repo only if it doesn't exist
if os.path.exists('/content/UniAD'):
    print("UniAD repository already exists, updating...")
    %cd UniAD
    
    # Check for local changes before resetting
    result = subprocess.run(['git', 'status', '--porcelain'], capture_output=True, text=True)
    if result.stdout.strip():
        print("\n⚠️  WARNING: You have local changes that will be preserved:")
        print(result.stdout)
        print("Stashing local changes...")
        !git stash
        stashed = True
    else:
        stashed = False
    
    !git fetch origin
    !git checkout v2.0-qianli
    !git reset --hard origin/v2.0-qianli
    
    # Restore stashed changes if any
    if stashed:
        print("\nRestoring your local changes...")
        !git stash pop
        print("✓ Local changes restored")
else:
    print("Cloning UniAD repository...")
    !git clone -b v2.0-qianli https://github.com/RobotMa/UniAD.git
    %cd UniAD

print(f"\nWorking directory: {os.getcwd()}")
!git branch --show-current

## 4. Install Dependencies

**First run:** Run all install cells in order. You may see a "Restart session" warning at the end — click it, then skip to Section 5.

**Subsequent runs:** If cache is restored successfully, you can skip the install cells.

In [ ]:
import os
import sys

# Cache paths
CACHE_DIR = "/content/drive/MyDrive/colab_cache/UniAD"
PACKAGES_CACHE = f"{CACHE_DIR}/site_packages_py312.tar.gz"
REPOS_CACHE = f"{CACHE_DIR}/repos.tar.gz"

# Create cache directory
!mkdir -p {CACHE_DIR}

# Set pip cache
os.environ["PIP_CACHE_DIR"] = f"{CACHE_DIR}/pip"

# Check if FULL cache exists
packages_cached = os.path.exists(PACKAGES_CACHE)
repos_cached = os.path.exists(REPOS_CACHE)

if packages_cached and repos_cached:
    print("="*60)
    print("FULL CACHE FOUND! Restoring everything...")
    print("="*60)
    
    print("\n[1/2] Restoring packages...")
    !tar -xzf {PACKAGES_CACHE} -C /usr/local/lib/python3.12/dist-packages/ 2>/dev/null
    
    print("[2/2] Restoring repos...")
    !tar -xzf {REPOS_CACHE} -C /content/ 2>/dev/null
    
    # Re-install editable packages (they need to be linked, not just extracted)
    print("\n[3/3] Re-linking editable packages...")
    %cd /content/mmcv
    !pip install -e . -q 2>/dev/null
    %cd /content/mmdetection
    !pip install -e . -q 2>/dev/null
    %cd /content/mmsegmentation
    !pip install -e . -q 2>/dev/null
    %cd /content/mmdetection3d
    !pip install -e . -q 2>/dev/null
    %cd /content/UniAD
    !pip install -e . -q 2>/dev/null
    
    # Comprehensive verification
    print("\nVerifying all dependencies...")
    try:
        import torch
        import mmcv
        import mmdet
        import mmseg
        import mmdet3d
        from nuscenes import NuScenes
        import numpy
        import einops
        print(f"✓ torch: {torch.__version__}")
        print(f"✓ mmcv: {mmcv.__version__}")
        print(f"✓ mmdet: {mmdet.__version__}")
        print(f"✓ mmseg: {mmseg.__version__}")
        print(f"✓ mmdet3d: {mmdet3d.__version__}")
        print(f"✓ numpy: {numpy.__version__}")
        print("✓ nuscenes-devkit: OK")
        print("✓ einops: OK")
        print("\nAll packages loaded successfully!")
        SKIP_INSTALL = True
    except ImportError as e:
        print(f"✗ Import error: {e}")
        print("Some packages missing. Run install cells below.")
        SKIP_INSTALL = False
    
    print("\n" + "="*60)
    print("DONE! Skip to Section 5 (Checkpoints)")
    print("="*60)
else:
    print("Cache not found. Will install and cache for next time.")
    if packages_cached:
        print("  - Packages: cached ✓")
    else:
        print("  - Packages: not cached")
    if repos_cached:
        print("  - Repos: cached ✓")
    else:
        print("  - Repos: not cached")
    print("\nRun the cells below to install dependencies.")
    SKIP_INSTALL = False

In [ ]:
# ============================================================
# FIRST RUN ONLY - Skip this cell if cache was restored above
# ============================================================

import os
import sys
import torch

# Check current environment (use Colab's pre-installed PyTorch)
print(f"Python version: {sys.version}")
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA version: {torch.version.cuda}")
!nvcc --version | grep release

print("\nUsing Colab's pre-installed PyTorch - no reinstall needed!")

# Build mmcv-full from source
print("\nBuilding mmcv-full from source (this takes ~10 min)...")
# Note: Using Colab's existing pip/setuptools to avoid restart warning
!pip install addict yapf "numpy>=1.22.4,<2.0" Pillow pyyaml

%cd /content
if not os.path.exists('/content/mmcv'):
    !git clone https://github.com/open-mmlab/mmcv.git -b v1.7.2
%cd mmcv
!MMCV_WITH_OPS=1 pip install -e . -v

In [ ]:
# Install mmdet and mmseg from source (compatible versions)
import os
%cd /content

# mmdetection 2.x
if not os.path.exists('/content/mmdetection'):
    !git clone https://github.com/open-mmlab/mmdetection.git -b v2.28.2
%cd mmdetection
!pip install -e . -v

# mmsegmentation
%cd /content
if not os.path.exists('/content/mmsegmentation'):
    !git clone https://github.com/open-mmlab/mmsegmentation.git -b v0.30.0
%cd mmsegmentation
!pip install -e . -v

# nuscenes-devkit
!pip install nuscenes-devkit motmetrics

In [ ]:
# Install mmdet3d from source
import os
%cd /content
if not os.path.exists('/content/mmdetection3d'):
    !git clone https://github.com/open-mmlab/mmdetection3d.git -b v1.0.0rc6
%cd mmdetection3d
!pip install -e . -v

# Install UniAD
%cd /content/UniAD
!pip install -r requirements.txt
!pip install -e .

print("\n" + "="*60)
print("Installation complete!")
print("Click 'Restart session' if prompted, then skip to Section 5.")
print("="*60)

In [ ]:
# ============================================================
# SAVE CACHE - Run after first successful install
# ============================================================

import os
import subprocess

CACHE_DIR = "/content/drive/MyDrive/colab_cache/UniAD"
PACKAGES_CACHE = f"{CACHE_DIR}/site_packages_py312.tar.gz"
REPOS_CACHE = f"{CACHE_DIR}/repos.tar.gz"

def run_tar_with_check(command, output_file, description):
    """Run tar command and verify success."""
    result = subprocess.run(command, shell=True, capture_output=True, text=True)
    if result.returncode != 0:
        print(f"✗ ERROR: Failed to save {description}")
        print(f"  Error: {result.stderr}")
        if "Disk quota exceeded" in result.stderr or "No space left" in result.stderr:
            print("  → Your Google Drive may be full. Free up space and try again.")
        return False
    
    # Verify file was created and has reasonable size
    if os.path.exists(output_file):
        size_mb = os.path.getsize(output_file) / (1024 * 1024)
        if size_mb < 10:  # Suspiciously small
            print(f"⚠️  WARNING: {description} is only {size_mb:.1f}MB (expected >100MB)")
            print("  The cache may be incomplete.")
            return False
        print(f"✓ {description}: {size_mb:.0f}MB")
        return True
    else:
        print(f"✗ ERROR: {output_file} was not created")
        return False

# Save packages cache
if not os.path.exists(PACKAGES_CACHE):
    print("[1/2] Saving packages to Drive (~5 min)...")
    success = run_tar_with_check(
        f"tar -czf {PACKAGES_CACHE} -C /usr/local/lib/python3.12/dist-packages/ .",
        PACKAGES_CACHE,
        "packages cache"
    )
    if not success:
        # Clean up partial file
        if os.path.exists(PACKAGES_CACHE):
            os.remove(PACKAGES_CACHE)
            print("  Removed incomplete cache file.")
else:
    size_mb = os.path.getsize(PACKAGES_CACHE) / (1024 * 1024)
    print(f"[1/2] Packages cache exists ✓ ({size_mb:.0f}MB)")

# Save repos cache (UniAD, mmcv, mmdetection, mmsegmentation, mmdetection3d)
if not os.path.exists(REPOS_CACHE):
    print("[2/2] Saving repos to Drive (~2 min)...")
    success = run_tar_with_check(
        f"tar -czf {REPOS_CACHE} -C /content/ UniAD mmcv mmdetection mmsegmentation mmdetection3d",
        REPOS_CACHE,
        "repos cache"
    )
    if not success:
        # Clean up partial file
        if os.path.exists(REPOS_CACHE):
            os.remove(REPOS_CACHE)
            print("  Removed incomplete cache file.")
else:
    size_mb = os.path.getsize(REPOS_CACHE) / (1024 * 1024)
    print(f"[2/2] Repos cache exists ✓ ({size_mb:.0f}MB)")

print("\n" + "="*60)
print("CACHE COMPLETE! Next time:")
print("1. Run cells 1-4 → Cache restored automatically")
print("2. Skip to Section 5 (Checkpoints)")
print("="*60)

## 5. Download Pretrained Checkpoints

In [ ]:
import os

%cd /content/UniAD
!mkdir -p ckpts

CKPT_CACHE = "/content/drive/MyDrive/colab_cache/UniAD_ckpts"

# Ensure checkpoint cache directory exists
!mkdir -p {CKPT_CACHE}

# Check if checkpoints already cached on Drive
if os.path.exists(f"{CKPT_CACHE}/uniad_base_track_map.pth"):
    print("Found cached checkpoints on Drive! Linking...")
    !ln -sf {CKPT_CACHE}/uniad_base_track_map.pth ckpts/
    !ln -sf {CKPT_CACHE}/bevformer_r101_dcn_24ep.pth ckpts/
else:
    print("Downloading checkpoints (will be cached to Drive)...")
    %cd {CKPT_CACHE}
    !wget -q --show-progress https://huggingface.co/OpenDriveLab/UniAD2.0_R101_nuScenes/resolve/main/ckpts/uniad_base_track_map.pth
    !wget -q --show-progress https://huggingface.co/OpenDriveLab/UniAD2.0_R101_nuScenes/resolve/main/ckpts/bevformer_r101_dcn_24ep.pth
    %cd /content/UniAD
    !ln -sf {CKPT_CACHE}/uniad_base_track_map.pth ckpts/
    !ln -sf {CKPT_CACHE}/bevformer_r101_dcn_24ep.pth ckpts/

print("\nCheckpoints ready:")
!ls -lh ckpts/

## 6. Setup nuScenes Dataset (Cached on Drive)

**Option A (Recommended):** Upload nuScenes to Google Drive manually:
1. Download from https://www.nuscenes.org/download (requires free registration)
2. Upload to `Google Drive/colab_cache/UniAD/nuscenes/`

**Option B (Mini dataset):** The cell below will download nuScenes mini via the official API (requires registration).

**Expected structure:**
```
Google Drive/colab_cache/UniAD/nuscenes/
├── maps/
├── samples/
├── sweeps/
├── v1.0-mini/ (or v1.0-trainval for full dataset)
```

**Important:** nuScenes mini is for testing only. For valid evaluation results matching the paper, use the full trainval dataset.

In [ ]:
import os

CACHE_DIR = "/content/drive/MyDrive/colab_cache/UniAD"
NUSCENES_CACHE = f"{CACHE_DIR}/nuscenes"

# Create data directory
!mkdir -p /content/UniAD/data

# Check if nuScenes exists on Drive
if os.path.exists(NUSCENES_CACHE) and os.listdir(NUSCENES_CACHE):
    print("✓ Found nuScenes on Google Drive!")
    !ln -sf {NUSCENES_CACHE} /content/UniAD/data/nuscenes
    print("  Linked to /content/UniAD/data/nuscenes")
    
    # Check contents
    print("\n  Contents:")
    !ls /content/UniAD/data/nuscenes/ | head -10
    
    # Warn if using mini dataset
    if os.path.exists(f"{NUSCENES_CACHE}/v1.0-mini") and not os.path.exists(f"{NUSCENES_CACHE}/v1.0-trainval"):
        print("\n⚠️  WARNING: Using nuScenes MINI dataset.")
        print("   Evaluation results will NOT match paper metrics.")
        print("   For valid results, upload the full trainval dataset.")
else:
    print("="*60)
    print("nuScenes dataset not found on Google Drive!")
    print("="*60)
    print("\nPlease download nuScenes manually:")
    print("1. Go to: https://www.nuscenes.org/download")
    print("2. Register for a free account")
    print("3. Download 'Mini' (for testing) or 'Trainval' (for full eval)")
    print("4. Upload to Google Drive at:")
    print(f"   {NUSCENES_CACHE}/")
    print("\nExpected structure after upload:")
    print("   nuscenes/")
    print("   ├── maps/")
    print("   ├── samples/")
    print("   ├── sweeps/")
    print("   └── v1.0-mini/ (or v1.0-trainval)")
    print("\nThen re-run this cell.")
    print("="*60)
    
    # Create the directory so user knows where to upload
    !mkdir -p {NUSCENES_CACHE}
    print(f"\nCreated empty directory: {NUSCENES_CACHE}")
    print("Upload your nuScenes data there.")

## 7. Prepare Data PKL Files (Auto-Cached)

PKL files contain preprocessed metadata (annotations, calibrations). They're generated once and cached to Drive.

In [ ]:
import os
import glob

%cd /content/UniAD

# Check if PKL files already exist (cached from previous run)
pkl_files = glob.glob("/content/UniAD/data/nuscenes/*_infos_*.pkl")

if pkl_files:
    print("✓ PKL files already exist (cached):")
    for f in pkl_files:
        print(f"  {os.path.basename(f)}")
else:
    print("✗ PKL files not found. Generating (one-time, ~10-20 min)...")
    print("  These will be cached to Drive for next time.\n")
    
    !python tools/create_data.py nuscenes \
        --root-path ./data/nuscenes \
        --out-dir ./data/nuscenes \
        --extra-tag nuscenes
    
    print("\n✓ PKL files generated and cached!")
    
    # Show generated files
    pkl_files = glob.glob("/content/UniAD/data/nuscenes/*_infos_*.pkl")
    for f in pkl_files:
        print(f"  {os.path.basename(f)}")

## 8. Run Evaluation

In [ ]:
import os

%cd /content/UniAD

# Detect dataset type and warn if using mini
NUSCENES_PATH = "/content/UniAD/data/nuscenes"
using_mini = os.path.exists(f"{NUSCENES_PATH}/v1.0-mini") and not os.path.exists(f"{NUSCENES_PATH}/v1.0-trainval")

if using_mini:
    print("="*60)
    print("⚠️  WARNING: Running evaluation on nuScenes MINI dataset")
    print("="*60)
    print("Results will NOT match paper metrics (AMOTA 0.394).")
    print("Mini dataset is for testing the pipeline only.")
    print("For valid evaluation, use the full trainval dataset.")
    print("="*60 + "\n")

# Run Stage 1 evaluation with 1 GPU
!python tools/test.py \
    projects/configs/stage1_track_map/base_track_map.py \
    ckpts/uniad_base_track_map.pth \
    --eval bbox

## Expected Results

If everything works correctly, you should see:
```
Aggregated results:
AMOTA    0.394
AMOTP    1.316
RECALL   0.484
```

## Troubleshooting

**Out of Memory:**
- Try reducing `queue_length` from 5 to 3 in the config
- Use Colab Pro for A100 GPU

**Missing files:**
- Ensure nuScenes dataset is properly linked
- Check that all .pkl files exist in data/nuscenes/

**Dependency errors:**
- Restart runtime and run cells from beginning
- If mmcv build fails, ensure you have GPU runtime enabled (Runtime → Change runtime type → T4 GPU)

**Session disconnected:**
- Colab sessions timeout after idle periods
- Your cached packages on Google Drive will persist — just re-run from Section 2